# AAV2 sélectivité — filtre par accord PROPORTIONNEL org2 ≈ org3

Alternative au filtre readout-depth (`AAV2_SEL_potts_readout_depth.ipynb`, seuil `T` absolu sur `compte_organoïde` ET `compte_virus`) et au filtre naïf intersection
(`AAV2_SEL_fitting_protocol_org2org3.ipynb`, juste `org2>0 ET org3>0`), sur demande explicite.

## Le problème avec un seuil absolu

Un seuil `T` absolu (`compte_organoïde ≥ T ET compte_virus ≥ T`) filtre par **abondance**, pas par **fiabilité** -- il élimine systématiquement les variants peu abondants, qu'ils soient bruités OU réellement peu sélectionnés (les "mauvais" variants). Pousser `T` plus haut pour gagner en fiabilité fait donc disparaître à la fois le bruit ET la partie basse du spectre réel de sélectivité -- la population de fit finit par ne plus représenter que les variants déjà les plus enrichis, ce qui appauvrit exactement le contraste bon/mauvais que la régression est censée apprendre.

## Le filtre proposé

$$|\text{compte\_organoïde\_2} - \text{compte\_organoïde\_3}| \;\le\; \frac{\text{compte\_organoïde\_2} + \text{compte\_organoïde\_3}}{5}$$

(en plus de `compte_organoïde_2 > 0 ET compte_organoïde_3 > 0`, pour que le ratio de sélectivité reste défini). La tolérance d'accord (`(org2+org3)/5`, soit ±20% de la somme des deux comptages) **grandit avec le comptage lui-même** -- ce n'est donc pas un seuil d'abondance : un couple `(org2=1, org3=1)` passe le filtre (accord parfait) exactement comme un couple `(org2=1000, org3=1050)` (accord à 5%), alors qu'un couple `(org2=1, org3=50)` est rejeté quel que soit le niveau d'abondance. Ça garde les variants **peu abondants mais dont les 2 réplicats se confirment mutuellement** -- y compris les "mauvais" variants (faible compte des deux côtés, en accord) -- et rejette les désaccords francs entre réplicats, à n'importe quelle échelle de comptage.

## Chiffres de cadrage (mesurés directement sur `AAV2_organoides.csv`, `plasmid≥1`)

- Intersection naïve (`org2>0 ET org3>0`, utilisée dans `AAV2_SEL_fitting_protocol_org2org3.ipynb`)   : **47 563** variants.
- Filtre proportionnel ci-dessus : **13 464** variants (28.3% de l'intersection naïve).
- Médiane `compte_organoïde_2/3` : 134.5/160.5 (intersection naïve) → 219.2/206.8 (filtre   proportionnel) -- remonte, mais BIEN MOINS agressivement qu'un seuil absolu haut.
- Variants à comptage très faible (`min(org2,org3)≤2`) : 24.6% de l'intersection naïve   conservés, contre seulement 12.7% du filtre proportionnel -- le filtre proportionnel retire   une partie des paires à faible comptage (celles qui NE S'ACCORDENT PAS), mais pas toutes   (celles qui s'accordent restent) -- contrairement à un seuil `T` absolu qui les retirerait   TOUTES sans distinction.

Même méthode de régression que partout ailleurs dans ce projet : Potts (F+J, 8541 features), poids inverse-variance `eps=0.5`, fit mutualisé org2+org3 (empilés comme observations séparées, F/J partagés). Cibles : `sel_org2`/`sel_org3` uniquement (org1 exclu, comme sur AAV5 et dans `AAV2_SEL_potts_readout_depth.ipynb`).

**⚠ 2026-09-18 : régression de Potts reconstruite avec le nouveau solveur par défaut du projet** (`RegressionV1.fit_weights_potts_from_data_matrixfree` -- matrix-free, plus de matrice de design dense matérialisée ; validé pour reproduire le solve classique quasi exactement, `r(F)`/`r(J) > 0.999999` sur données AAV2 réelles -- cf. `viability/AAV2_potts_ridge_matrixfree_validation.ipynb`). Même signature d'appel/retour que l'ancienne `fit_weights_potts_from_data` (`rank` vaut toujours `None` ici -- pas de diagnostic SVD avec un solveur itératif). L'ancienne version de ce notebook (solveur dense/SVD) est archivée dans `obsolete_dense_matrix_potts_regression/`. **Sorties de cellules effacées** (méthode de fit changée, anciens chiffres plus valides) -- à ré-exécuter. Notebook préparé mais **jamais exécuté par Claude** (`feedback_user_runs_notebooks`).

### 0. Setup + population (filtre d'accord proportionnel)

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
LIB = _root / "lib"

import RegressionV1 as R
from analysisV1 import AA_LABELS, pearson, topk_recovery
assert "Modelization_V2" in R.__file__, R.__file__
print(R.message)

CSV = Path("AAV2_organoides.csv")
if not CSV.exists():
    CSV = _root / "notebooks/notebooks/AAVs dataset/AAV2/AAV2_organoides.csv"
assert CSV.exists(), CSV

viab_col = "log2_enrichissement_virus_sur_plasmide"
sel = {2: "log2_enrichissement_organoide_2_adn_sur_virus",
       3: "log2_enrichissement_organoide_3_adn_sur_virus"}
cnt = {i: f"compte_organoide_{i}_adn" for i in (2, 3)}
use = ["sequence", "compte_plasmide", "compte_virus", viab_col, *cnt.values(), *sel.values()]
df_raw = pd.read_csv(CSV, usecols=use,
                      dtype={c: "float32" for c in use if c != "sequence"} | {"sequence": "string"})
df_raw["sequence"] = df_raw["sequence"].astype("string")
print(f"CSV brut : {df_raw.shape[0]:,} lignes")

plasmid = df_raw["compte_plasmide"].to_numpy(np.float64)
virus   = df_raw["compte_virus"].to_numpy(np.float64)
org2    = df_raw[cnt[2]].to_numpy(np.float64)
org3    = df_raw[cnt[3]].to_numpy(np.float64)

PLASMID_MIN = 1   # meme filtre qualite-viab minimal que le reste du projet AAV2 (pas de cap ratio)
viab_keep = plasmid >= PLASMID_MIN

both_present = (org2 > 0) & (org3 > 0)
naive_inter = viab_keep & both_present

S = org2 + org3
TOLERANCE_FRAC = 1.0 / 5.0   # |org2-org3| <= (org2+org3)/5, sur demande explicite
prop_keep = viab_keep & both_present & (np.abs(org2 - org3) <= TOLERANCE_FRAC * S)

print(f"intersection naive (org2>0 ET org3>0, plasmid>={PLASMID_MIN})     : {int(naive_inter.sum()):,}")
print(f"filtre proportionnel (|org2-org3|<=(org2+org3)/{int(1/TOLERANCE_FRAC)})       : "
      f"{int(prop_keep.sum()):,}  ({prop_keep.sum() / naive_inter.sum():.1%} de l'intersection naive)")

low = (org2 <= 2) | (org3 <= 2)
print(f"  dont min(org2,org3)<=2  -- naive : {int((naive_inter & low).sum()):,}/{int(naive_inter.sum()):,} "
      f"({(naive_inter & low).sum() / naive_inter.sum():.1%})  |  "
      f"proportionnel : {int((prop_keep & low).sum()):,}/{int(prop_keep.sum()):,} "
      f"({(prop_keep & low).sum() / prop_keep.sum():.1%})")

df_full = df_raw.loc[prop_keep].reset_index(drop=True)
print(f"\npopulation retenue pour ce notebook : {df_full.shape[0]:,} lignes")

L, A = 7, 20
lut = np.zeros(256, np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i
seq_matrix = lut[np.frombuffer("".join(df_full["sequence"]).encode("ascii"), np.uint8)].reshape(len(df_full), L)


def score_FJ(seq, F, J):
    F, J = np.asarray(F), np.asarray(J)
    Fp = F[seq, np.arange(L)].sum(axis=1).astype(np.float64)
    Jp = np.zeros(len(seq), np.float64)
    for i in range(L):
        for j in range(i + 1, L):
            Jp += J[i, j, seq[:, i], seq[:, j]]
    return Fp + Jp


def obs_weight(num, den):
    return 1.0 / (1.0 / (num + 0.5) + 1.0 / (den + 0.5))


def spearman(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return pearson(np.argsort(np.argsort(a)), np.argsort(np.argsort(b)))

### 1. Plafond de reproductibilité r(y2, y3) sur cette population

Un seul nombre ici (pas de sweep -- le filtre n'a pas de paramètre `T` à balayer) : jusqu'où un fit peut espérer monter sur cette population précise.

In [ ]:
y2f = df_full[sel[2]].to_numpy(np.float64)
y3f = df_full[sel[3]].to_numpy(np.float64)
m_ceil = np.isfinite(y2f) & np.isfinite(y3f)
r_ceiling = pearson(y2f[m_ceil], y3f[m_ceil])
print(f"r(y_org2, y_org3) sur les {int(m_ceil.sum()):,} variants du filtre proportionnel : {r_ceiling:+.3f}")

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.hexbin(y2f[m_ceil], y3f[m_ceil], gridsize=45, bins="log", cmap="viridis", mincnt=1)
lo, hi = min(y2f[m_ceil].min(), y3f[m_ceil].min()), max(y2f[m_ceil].max(), y3f[m_ceil].max())
ax.plot([lo, hi], [lo, hi], "w--", lw=0.8)
ax.set(title=f"AAV2 -- y_org2 vs y_org3, filtre proportionnel (r={r_ceiling:+.3f}, n={int(m_ceil.sum()):,})",
       xlabel="log2 enrichment org2", ylabel="log2 enrichment org3")
plt.tight_layout(); plt.show()

### 2. Fit mutualisé org2 + org3

Même recette que la section 6 d'`AAV2_SEL_potts_readout_depth.ipynb` : les lignes de org2 et org3 empilées comme observations séparées (F/J partagés), split 50/50, poids inverse-variance `eps=0.5`. **λ=0** (OLS minimum-norm, poids de production -- convention du projet pour ces fits à n modeste face à 8541 features) **et** la ridge à λ choisi par CV, comparés.

In [ ]:
i2 = np.flatnonzero(np.isfinite(y2f))
i3 = np.flatnonzero(np.isfinite(y3f))
w2f = obs_weight(df_full[cnt[2]].to_numpy(np.float64), virus[prop_keep])
w3f = obs_weight(df_full[cnt[3]].to_numpy(np.float64), virus[prop_keep])

tr2, te2 = train_test_split(i2, test_size=0.5, random_state=0)
tr3, te3 = train_test_split(i3, test_size=0.5, random_state=0)
S_pool = np.vstack([seq_matrix[tr2], seq_matrix[tr3]])
y_pool = np.concatenate([y2f[tr2], y3f[tr3]])
w_pool = np.concatenate([w2f[tr2], w3f[tr3]])
print(f"pool de fit : {len(S_pool):,} observations (org2 {len(tr2):,} + org3 {len(tr3):,})")

Fp, Jp, rankp, _ = R.fit_weights_potts_from_data_matrixfree(S_pool, y_pool, sample_weight=w_pool, verbose=False, lam=0.0)
Fp, Jp = np.asarray(Fp), np.asarray(Jp)

LAMBDAS_CV = np.logspace(-4, 8, 21)
print(f"CV pool  ({len(LAMBDAS_CV)} λ × 5 folds)...")
Fpc, Jpc, _, infoc = R.fit_weights_potts_from_data_matrixfree(
    S_pool, y_pool, sample_weight=w_pool, lambdas_grid=LAMBDAS_CV, k_folds=5, seed=0, verbose=True)
Fpc, Jpc = np.asarray(Fpc), np.asarray(Jpc)
lam_pool = infoc["lam"]

print(f"\nfit mutualise org2+org3  (n={len(S_pool):,})")
for tag, F_, J_ in [("λ=0", Fp, Jp), (f"CV λ={lam_pool:.3g}", Fpc, Jpc)]:
    r2 = pearson(y2f[te2], score_FJ(seq_matrix[te2], F_, J_))
    r3 = pearson(y3f[te3], score_FJ(seq_matrix[te3], F_, J_))
    print(f"  [{tag:14s}] held-out r : org2 {r2:+.3f} | org3 {r3:+.3f}")
print(f"  plafond r(y2,y3) : {r_ceiling:+.3f}")

USE_CV = (pearson(y2f[te2], score_FJ(seq_matrix[te2], Fpc, Jpc))
          + pearson(y3f[te3], score_FJ(seq_matrix[te3], Fpc, Jpc))
          >= pearson(y2f[te2], score_FJ(seq_matrix[te2], Fp, Jp))
          + pearson(y3f[te3], score_FJ(seq_matrix[te3], Fp, Jp)))
F_final, J_final = (Fpc, Jpc) if USE_CV else (Fp, Jp)
tag_final = "cv" if USE_CV else "unreg"
print(f"\n-> retenu : {tag_final}")

### 3. Scatter -- score prédit vs log2 enrichment réel (held-out)

Même style que section 7 d'`AAV2_SEL_potts_readout_depth.ipynb` : hexbin + moyenne de `y` par bin de score ±σ + diagonale + Pearson `r`/Spearman `ρ`.

In [ ]:
def scatter_score_vs_y(ax, pred, y, title, nbin=18):
    pred, y = np.asarray(pred), np.asarray(y)
    hb = ax.hexbin(pred, y, gridsize=50, bins="log", cmap="viridis", mincnt=1)
    edges = np.quantile(pred, np.linspace(0, 1, nbin + 1))
    cx, cy, ce = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pred >= lo) & (pred <= hi)
        if m.sum() >= 20:
            cx.append(pred[m].mean()); cy.append(y[m].mean()); ce.append(y[m].std())
    ax.errorbar(cx, cy, yerr=ce, fmt="o-", color="crimson", ms=4, lw=1.3, capsize=2,
                label="moy(y) par bin de score ±σ")
    lo, hi = min(pred.min(), y.min()), max(pred.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=0.9)
    r, rho = pearson(y, pred), spearman(pred, y)
    ax.set(title=f"{title}\nr = {r:+.3f}   ρ_Spearman = {rho:+.3f}   n = {len(y):,}",
           xlabel="score prédit  (F + J)", ylabel="log2 enrichment réel (held-out)")
    ax.legend(fontsize=7, loc="upper left")
    return hb


fig, axes = plt.subplots(1, 2, figsize=(13, 5.3))
for ax, (label, pred, yv) in zip(axes, [
    ("pool → org2 held-out", score_FJ(seq_matrix[te2], F_final, J_final), y2f[te2]),
    ("pool → org3 held-out", score_FJ(seq_matrix[te3], F_final, J_final), y3f[te3]),
]):
    hb = scatter_score_vs_y(ax, pred, yv, label)
    fig.colorbar(hb, ax=ax, label="variants (log)")
fig.suptitle(f"AAV2 sélectivité -- filtre proportionnel -- score Potts vs log2 enrichment réel ({tag_final})")
plt.tight_layout(); plt.show()

### 3a. Test après-coup sur TOUTES les données (fit principal, pas le filtre)

Le `r` held-out des sections 2/3 ne mesure la généralisation qu'À L'INTÉRIEUR de la population déjà filtrée — `te2`/`te3` sont des indices dans `df_full`, jamais dans les variants rejetés par `plasmid_min`/`tolerance_frac`. Ici : `F_final`/`J_final` (le fit **principal** de la section 2 — pas le sweep de la section 6) sont utilisés pour scorer **tout le CSV brut** `df_raw` (4 273 463 lignes), avec pour seule contrainte que `y2`/`y3` soit fini (`compte_organoide_i > 0`, pas le filtre proportionnel). Même convention que `r_sorted` vs `r_brut` dans `AAV2_potts_regression.ipynb` §11 : un fit qui surapprend sur son sous-ensemble sans généraliser au brut se voit dans l'écart entre les deux.

In [ ]:
seq_all = lut[np.frombuffer("".join(df_raw["sequence"]).encode("ascii"), np.uint8)].reshape(len(df_raw), L)
y2_all = df_raw[sel[2]].to_numpy(np.float64)
y3_all = df_raw[sel[3]].to_numpy(np.float64)

m2_all = np.isfinite(y2_all)
m3_all = np.isfinite(y3_all)

score2_all = score_FJ(seq_all[m2_all], F_final, J_final)
score3_all = score_FJ(seq_all[m3_all], F_final, J_final)
r_brut2 = pearson(y2_all[m2_all], score2_all)
r_brut3 = pearson(y3_all[m3_all], score3_all)

r_heldout2 = pearson(y2f[te2], score_FJ(seq_matrix[te2], F_final, J_final))
r_heldout3 = pearson(y3f[te3], score_FJ(seq_matrix[te3], F_final, J_final))

print(f"r(score, reel) sur TOUT le CSV brut (pas le filtre)   : org2 {r_brut2:+.3f} (n={int(m2_all.sum()):,})  "
      f"| org3 {r_brut3:+.3f} (n={int(m3_all.sum()):,})")
print(f"r(score, reel) sur la population filtree (held-out)   : org2 {r_heldout2:+.3f} "
      f"| org3 {r_heldout3:+.3f}  (n={len(df_full):,} variants dans la population filtree, "
      f"sur {len(df_raw):,} au total dans le brut)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.3))
for ax, (label, pred, yv) in zip(axes, [
    ("TOUT le CSV brut -- org2", score2_all, y2_all[m2_all]),
    ("TOUT le CSV brut -- org3", score3_all, y3_all[m3_all]),
]):
    hb = scatter_score_vs_y(ax, pred, yv, label)
    fig.colorbar(hb, ax=ax, label="variants (log)")
fig.suptitle("AAV2 selectivite -- fit principal (section 2) score vs reel, sur TOUTES les donnees (pas le filtre)")
plt.tight_layout(); plt.show()

### 3a-bis. `y2` vs `y3` — log2 enrichment organoïde, tout le dataset

Correction : pas les comptages bruts, la **cible de sélectivité** elle-même (`y2`/`y3` = `log2_enrichissement_organoide_{2,3}_adn_sur_virus`, déjà en log2 dans le CSV) — l'équivalent du plafond de reproductibilité `r_ceiling` de la section 1, mais sur **tout le dataset brut** (`y2_all`/`y3_all`, section 3a) au lieu de la seule population filtrée `df_full`. Seule contrainte : les deux valeurs finies (`org2>0 ET org3>0`).

In [ ]:
m_ceil_all = np.isfinite(y2_all) & np.isfinite(y3_all)
r_ceiling_all = pearson(y2_all[m_ceil_all], y3_all[m_ceil_all])

fig, ax = plt.subplots(figsize=(6.5, 6))
hb = ax.hexbin(y2_all[m_ceil_all], y3_all[m_ceil_all], gridsize=60, bins="log", cmap="viridis", mincnt=1)
lo, hi = min(y2_all[m_ceil_all].min(), y3_all[m_ceil_all].min()), max(y2_all[m_ceil_all].max(), y3_all[m_ceil_all].max())
ax.plot([lo, hi], [lo, hi], "w--", lw=0.8)
ax.set(title=f"AAV2 -- log2 enrichment org2 vs org3, tout le dataset\n"
             f"r={r_ceiling_all:+.3f}  n={int(m_ceil_all.sum()):,}",
       xlabel="log2 enrichment org2 (organoide_2_adn/virus)", ylabel="log2 enrichment org3 (organoide_3_adn/virus)")
fig.colorbar(hb, ax=ax, label="variants (log)")
plt.tight_layout(); plt.show()

print(f"r(y2, y3) sur TOUT le dataset (org2>0 ET org3>0, pas le filtre proportionnel) : {r_ceiling_all:+.3f}  "
      f"(n={int(m_ceil_all.sum()):,})")
print(f"  (pour reference : r_ceiling sur la population filtree, section 1 = {r_ceiling:+.3f}, "
      f"n={int(m_ceil.sum()):,})")

### 3b. `ShallowProfileMLP` — même split, comparaison directe au Potts

Même architecture/boucle d'entraînement que partout ailleurs dans le projet (`AAV2_viab_top10k_potts_protocol_mlp.ipynb`, `AAV5_SEL_profile_model_*.ipynb`) : one-hot 140 → MLP (128→64→1, dropout 0.1, BatchNorm) → 1 scalaire, MSE **non pondérée** (la pondération inverse-variance dans la loss a été testée et écartée ailleurs dans ce projet, `AAV5_SEL_profile_model_sel_org2_invvar.ipynb`, dominée par quelques poids extrêmes issus de contamination — pas retentée ici).

Entraîné sur **exactement le même pool** que le fit Potts (`S_pool`/`y_pool`, org2+org3 empilés), évalué sur les **mêmes held-out** `te2`/`te3` — comparaison directe, ligne par ligne, avec `r_final`/`tag_final` de la section 2.

In [ ]:
import jax.numpy as jnp
from flax import nnx
from typing import Optional
import optax
from tqdm.auto import tqdm


class ShallowProfileMLP(nnx.Module):
    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x, *, train: bool, rngs: Optional[nnx.Rngs] = None):
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        return self.linear3(x).squeeze(-1)


@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    return jnp.mean((model(x, train=False) - y) ** 2)


@nnx.jit
def predict_step(model, x):
    return model(x, train=False)


@nnx.scan(in_axes=(nnx.Carry, 0, 0), out_axes=(nnx.Carry, 0))
def train_epoch_scan(carry, xb, yb):
    model, optimizer, rngs = carry
    loss = train_step(model, optimizer, xb, yb, rngs)
    return (model, optimizer, rngs), loss


def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def train_mlp(model_cls, model_kwargs, X_train, y_train, X_val, y_val,
              epochs=300, batch_size=256, peak_lr=1e-3, final_lr=1e-5,
              weight_decay=0, patience=8, seed=0):
    rngs  = nnx.Rngs(seed)
    model = model_cls(rngs=rngs, **model_kwargs)
    n_train = X_train.shape[0]
    batch_size = min(batch_size, max(n_train, 1))
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps = steps_per_epoch * epochs
    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr, warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9), end_value=final_lr)
    optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param)
    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val, y_val = jnp.asarray(X_val), jnp.asarray(y_val)
    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}
    for epoch in tqdm(range(epochs), desc="  epochs", leave=False):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm = jax.random.permutation(perm_key, n_train)
        batch_idx = perm[: steps_per_epoch * batch_size].reshape(steps_per_epoch, batch_size)
        (model, optimizer, rngs), step_losses = train_epoch_scan((model, optimizer, rngs), X_train[batch_idx], y_train[batch_idx])
        val_loss = float(eval_step(model, X_val, y_val))
        history["train_loss"].append(float(jnp.mean(step_losses)))
        history["val_loss"].append(val_loss)
        if val_loss < best_val - 1e-5:
            best_val, bad_epochs, best_state = val_loss, 0, nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    nnx.update(model, best_state)
    return model, history


import jax
oh = lambda seq_idx: np.eye(20, dtype=np.float32)[seq_idx].reshape(len(seq_idx), -1)

X_pool = oh(S_pool)
Xtr, ytr, Xva, yva = split_train_val(X_pool, y_pool.astype(np.float32), val_frac=0.15, seed=0)
print(f"train={len(Xtr):,}  val={len(Xva):,}  (meme pool que le fit Potts, n={len(X_pool):,})")

mlp, hist = train_mlp(ShallowProfileMLP, dict(input_dim=7 * 20), Xtr, ytr, Xva, yva, seed=0)
print(f"-> {len(hist['train_loss'])} epochs (early stopping, patience=8)")

X_te2, X_te3 = oh(seq_matrix[te2]), oh(seq_matrix[te3])
pred_mlp2 = np.asarray(predict_step(mlp, jnp.asarray(X_te2)))
pred_mlp3 = np.asarray(predict_step(mlp, jnp.asarray(X_te3)))
r_mlp2 = pearson(y2f[te2], pred_mlp2)
r_mlp3 = pearson(y3f[te3], pred_mlp3)
print(f"held-out r (MLP) : org2 {r_mlp2:+.3f} | org3 {r_mlp3:+.3f}")
print(f"held-out r (Potts, {tag_final}) : org2 {pearson(y2f[te2], score_FJ(seq_matrix[te2], F_final, J_final)):+.3f} "
      f"| org3 {pearson(y3f[te3], score_FJ(seq_matrix[te3], F_final, J_final)):+.3f}")
print(f"plafond r(y2,y3) : {r_ceiling:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.3))
for ax, (label, pred, yv) in zip(axes, [
    ("pool -> org2 held-out (MLP)", pred_mlp2, y2f[te2]),
    ("pool -> org3 held-out (MLP)", pred_mlp3, y3f[te3]),
]):
    hb = scatter_score_vs_y(ax, pred, yv, label)
    ax.set_xlabel("log2 enrichment predit (MLP)")
    fig.colorbar(hb, ax=ax, label="variants (log)")
fig.suptitle("AAV2 selectivite -- filtre proportionnel -- ShallowProfileMLP vs log2 enrichment reel")
plt.tight_layout(); plt.show()

### 4. Export

In [ ]:
np.save(LIB / "aav2_F_sel_pool_potts_proportional_unreg.npy", F_final.astype(np.float32))
np.save(LIB / "aav2_J_sel_pool_potts_proportional_unreg.npy", J_final.astype(np.float32))
print("-> lib/aav2_F_sel_pool_potts_proportional_unreg.npy / aav2_J_sel_pool_potts_proportional_unreg.npy")
print(f"(fit={tag_final}, filtre |org2-org3|<=(org2+org3)/{int(1/TOLERANCE_FRAC)}, n={len(df_full):,} variants)")

### 5. Conclusion

- **§0** : le filtre proportionnel garde 13 464 variants sur l'intersection naïve de 47 563   (28.3%) -- une réduction bien plus douce qu'un seuil `T` absolu élevé, et qui ne retire pas   systématiquement les variants à faible comptage (seulement ceux dont les 2 réplicats se   contredisent).
- **§1** : plafond `r(y2,y3)` sur cette population -- à comparer directement au plafond   équivalent de `AAV2_SEL_potts_readout_depth.ipynb` à `T` comparable en taille de population.
- **§2/§3** : fit mutualisé + held-out -- si ce `r` est proche du plafond §1, le filtre   proportionnel capture l'essentiel du signal reproductible sans sacrifier la partie basse du   spectre de sélectivité.

Poids exportés : `lib/aav2_{F,J}_sel_pool_potts_proportional_unreg.npy`. Comparaison directe avec les poids `readoutT5` (`AAV2_SEL_potts_readout_depth.ipynb`) faite dans `AAV2_SEL_fitting_protocol_org2org3.ipynb`, nouvelle section ajoutée à cet effet.

**Notebook jamais exécuté par Claude** (`feedback_user_runs_notebooks`) -- logique validée par smoke-test sur un sous-échantillon des données réelles.

### 6. Sweep — `plasmid_min` × `tolerance_frac`

Rejoue les sections 0/1/2/3b (population, fit Potts, fit MLP) pour une grille de filtres au lieu du seul couple fixé en section 0 :

- `PLASMID_MIN_GRID = [1, 2, 3, 5, 10]`
- `TOLERANCE_DENOM_GRID = [1, 2, 3, 4, 5, 10]` (`tolerance_frac = 1/denom` — `denom=1` équivaut à `|org2-org3| ≤ org2+org3`, toujours vrai pour des comptages positifs, donc **pas de contrainte d'accord du tout** : c'est l'intersection naïve `org2>0 ET org3>0`, la référence la plus permissive de la grille)

30 combinaisons. **Potts** : `λ=0` uniquement (pas de CV) — même convention que les autres sweeps du projet (`AAV2_potts_regression.ipynb` §11, `AAV5_SEL_potts_readout_depth.ipynb`), refaire une CV 5-fold 30 fois serait beaucoup trop lent. **MLP** : budget réduit (`epochs=60, patience=6` au lieu du défaut `300/8` de la section 3b) pour la même raison de tractabilité — un compromis vitesse/qualité assumé pour ce balayage, pas la config "pleine puissance" d'un fit isolé. Combinaisons trop petites (moins de 50 lignes retenues, ou moins de 20 lignes finies sur un réplicat) sautées avec `NaN`.

In [ ]:
PLASMID_MIN_GRID = [1, 2, 3, 5, 10]
TOLERANCE_DENOM_GRID = [1, 2, 3, 4, 5, 10]
MIN_ROWS = 50
MIN_FINITE_PER_REPLICATE = 20


def build_population_mask(plasmid_min, tol_denom):
    viab_keep_s = plasmid >= plasmid_min
    tol_frac_s = 1.0 / tol_denom
    return viab_keep_s & both_present & (np.abs(org2 - org3) <= tol_frac_s * S)


def fit_potts_on_mask(keep_mask, seed=0):
    df_s = df_raw.loc[keep_mask].reset_index(drop=True)
    n_s = len(df_s)
    if n_s < MIN_ROWS:
        return None

    seq_s = lut[np.frombuffer("".join(df_s["sequence"]).encode("ascii"), np.uint8)].reshape(n_s, L)
    y2_s = df_s[sel[2]].to_numpy(np.float64)
    y3_s = df_s[sel[3]].to_numpy(np.float64)
    virus_s = df_s["compte_virus"].to_numpy(np.float64)
    w2_s = obs_weight(df_s[cnt[2]].to_numpy(np.float64), virus_s)
    w3_s = obs_weight(df_s[cnt[3]].to_numpy(np.float64), virus_s)

    i2_s = np.flatnonzero(np.isfinite(y2_s))
    i3_s = np.flatnonzero(np.isfinite(y3_s))
    if len(i2_s) < MIN_FINITE_PER_REPLICATE or len(i3_s) < MIN_FINITE_PER_REPLICATE:
        return None

    tr2_s, te2_s = train_test_split(i2_s, test_size=0.5, random_state=seed)
    tr3_s, te3_s = train_test_split(i3_s, test_size=0.5, random_state=seed)
    S_pool_s = np.vstack([seq_s[tr2_s], seq_s[tr3_s]])
    y_pool_s = np.concatenate([y2_s[tr2_s], y3_s[tr3_s]])
    w_pool_s = np.concatenate([w2_s[tr2_s], w3_s[tr3_s]])

    F_s, J_s, rank_s, _ = R.fit_weights_potts_from_data_matrixfree(
        S_pool_s, y_pool_s, sample_weight=w_pool_s, verbose=False, lam=0.0)
    F_s, J_s = np.asarray(F_s), np.asarray(J_s)

    r_potts2 = pearson(y2_s[te2_s], score_FJ(seq_s[te2_s], F_s, J_s))
    r_potts3 = pearson(y3_s[te3_s], score_FJ(seq_s[te3_s], F_s, J_s))

    m_ceil_s = np.isfinite(y2_s) & np.isfinite(y3_s)
    r_ceil_s = pearson(y2_s[m_ceil_s], y3_s[m_ceil_s]) if m_ceil_s.sum() >= MIN_FINITE_PER_REPLICATE else np.nan

    return dict(n=n_s, rank=rank_s, n_pool=len(S_pool_s), r_ceiling=r_ceil_s,
                r_potts2=r_potts2, r_potts3=r_potts3,
                S_pool=S_pool_s, y_pool=y_pool_s,
                seq_s=seq_s, y2_s=y2_s, y3_s=y3_s, te2_s=te2_s, te3_s=te3_s)


def fit_mlp_on_result(res, epochs=60, patience=6, seed=0):
    X_pool_s = oh(res["S_pool"])
    Xtr_s, ytr_s, Xva_s, yva_s = split_train_val(X_pool_s, res["y_pool"].astype(np.float32), val_frac=0.15, seed=seed)
    mlp_s, _ = train_mlp(ShallowProfileMLP, dict(input_dim=7 * 20), Xtr_s, ytr_s, Xva_s, yva_s,
                          epochs=epochs, patience=patience, seed=seed)
    X_te2_s = oh(res["seq_s"][res["te2_s"]])
    X_te3_s = oh(res["seq_s"][res["te3_s"]])
    pred2 = np.asarray(predict_step(mlp_s, jnp.asarray(X_te2_s)))
    pred3 = np.asarray(predict_step(mlp_s, jnp.asarray(X_te3_s)))
    r_mlp2 = pearson(res["y2_s"][res["te2_s"]], pred2)
    r_mlp3 = pearson(res["y3_s"][res["te3_s"]], pred3)
    return r_mlp2, r_mlp3


sweep_rows = []
for pm in PLASMID_MIN_GRID:
    for denom in TOLERANCE_DENOM_GRID:
        keep_s = build_population_mask(pm, denom)
        n_s = int(keep_s.sum())
        res = fit_potts_on_mask(keep_s)
        if res is None:
            print(f"plasmid_min={pm:<3} tol=1/{denom:<3}  n={n_s:>7,}  -- trop peu de donnees, saute")
            sweep_rows.append(dict(plasmid_min=pm, tol_denom=denom, n=n_s, rank=np.nan,
                                    r_ceiling=np.nan, r_potts2=np.nan, r_potts3=np.nan,
                                    r_mlp2=np.nan, r_mlp3=np.nan))
            continue

        r_mlp2, r_mlp3 = fit_mlp_on_result(res)
        print(f"plasmid_min={pm:<3} tol=1/{denom:<3}  n={res['n']:>7,} pool={res['n_pool']:>6,} "
              f"ceiling={res['r_ceiling']:+.3f}  "
              f"Potts org2/org3={res['r_potts2']:+.3f}/{res['r_potts3']:+.3f}  "
              f"MLP org2/org3={r_mlp2:+.3f}/{r_mlp3:+.3f}")
        sweep_rows.append(dict(plasmid_min=pm, tol_denom=denom, n=res["n"], rank=res["rank"],
                                r_ceiling=res["r_ceiling"], r_potts2=res["r_potts2"], r_potts3=res["r_potts3"],
                                r_mlp2=r_mlp2, r_mlp3=r_mlp3))

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df)

In [ ]:
sweep_df["r_potts_mean"] = sweep_df[["r_potts2", "r_potts3"]].mean(axis=1)
sweep_df["r_mlp_mean"] = sweep_df[["r_mlp2", "r_mlp3"]].mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, col, title, fmt in zip(
    axes,
    ["r_potts_mean", "r_mlp_mean", "n"],
    ["Potts (moy org2/org3, held-out)", "MLP (moy org2/org3, held-out)", "n variants retenus"],
    ["{:.2f}", "{:.2f}", "{:,.0f}"],
):
    piv = sweep_df.pivot(index="plasmid_min", columns="tol_denom", values=col)
    piv = piv.reindex(index=PLASMID_MIN_GRID, columns=TOLERANCE_DENOM_GRID)
    im = ax.imshow(piv.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(TOLERANCE_DENOM_GRID)))
    ax.set_xticklabels([f"1/{d}" for d in TOLERANCE_DENOM_GRID])
    ax.set_yticks(range(len(PLASMID_MIN_GRID)))
    ax.set_yticklabels(PLASMID_MIN_GRID)
    ax.set_xlabel("tolerance_frac")
    ax.set_ylabel("plasmid_min")
    ax.set_title(title)
    for yi in range(piv.shape[0]):
        for xi in range(piv.shape[1]):
            v = piv.values[yi, xi]
            if np.isfinite(v):
                ax.text(xi, yi, fmt.format(v), ha="center", va="center", color="white", fontsize=7)
    fig.colorbar(im, ax=ax)
fig.suptitle("AAV2 sélectivité -- sweep plasmid_min x tolerance_frac (Potts lam=0 vs MLP, meme split par cellule)")
plt.tight_layout(); plt.show()

best_potts = sweep_df.loc[sweep_df["r_potts_mean"].idxmax()]
best_mlp = sweep_df.loc[sweep_df["r_mlp_mean"].idxmax()]
print(f"meilleur Potts : plasmid_min={best_potts['plasmid_min']:.0f}  tol=1/{best_potts['tol_denom']:.0f}  "
      f"r_mean={best_potts['r_potts_mean']:+.3f}  n={best_potts['n']:.0f}")
print(f"meilleur MLP   : plasmid_min={best_mlp['plasmid_min']:.0f}  tol=1/{best_mlp['tol_denom']:.0f}  "
      f"r_mean={best_mlp['r_mlp_mean']:+.3f}  n={best_mlp['n']:.0f}")

### 7. Quel filtre généralise le mieux sur TOUT le dataset brut ?

Chaque `r_potts2`/`r_potts3` de la section 6 ne mesure la généralisation qu'À L'INTÉRIEUR de la
population déjà filtrée par ce même filtre (`te2_s`/`te3_s` sont des indices dans le sous-ensemble
retenu, jamais dans les variants qu'il a rejetés) — un filtre très restrictif peut afficher un bon
`r` held-out tout en apprenant un score qui ne généralise pas du tout en dehors de son propre
filtre (cf. la section 6 : `r_ceiling` explose de +0.49 à +0.99 entre `tol=1/1` et `tol=1/5`, sans
que `r_potts` s'améliore systématiquement — signe que le held-out à l'intérieur d'un filtre serré
ne dit pas tout).

Ici : **même critère pour tout le monde** — refit chaque combinaison `plasmid_min × tolerance_frac`
de la grille (même recette `lam=0` que la section 6, mêmes seeds → mêmes populations, mais on garde
`F_s`/`J_s` cette fois), puis score sur **TOUT le CSV brut** (`y2_all`/`y3_all`, section 3a —
72 671 variants `org2>0 ET org3>0`, indépendamment de ce que chaque filtre a retenu). Comparé aussi
au fit **principal** de ce notebook (`tol=1/5`, section 2/3a, déjà scoré sur le brut : `r_brut2`/
`r_brut3`) et au fit **readout-depth `T=5`** (`AAV2_SEL_potts_readout_depth.ipynb`, poids chargés
directement depuis `lib/` s'ils existent).

In [ ]:
def fit_potts_weights_on_mask(keep_mask, seed=0):
    """Meme recette que fit_potts_on_mask (section 6), mais retourne F_s/J_s au lieu de les jeter."""
    df_s = df_raw.loc[keep_mask].reset_index(drop=True)
    n_s = len(df_s)
    if n_s < MIN_ROWS:
        return None

    seq_s = lut[np.frombuffer("".join(df_s["sequence"]).encode("ascii"), np.uint8)].reshape(n_s, L)
    y2_s = df_s[sel[2]].to_numpy(np.float64)
    y3_s = df_s[sel[3]].to_numpy(np.float64)
    virus_s = df_s["compte_virus"].to_numpy(np.float64)
    w2_s = obs_weight(df_s[cnt[2]].to_numpy(np.float64), virus_s)
    w3_s = obs_weight(df_s[cnt[3]].to_numpy(np.float64), virus_s)

    i2_s = np.flatnonzero(np.isfinite(y2_s))
    i3_s = np.flatnonzero(np.isfinite(y3_s))
    if len(i2_s) < MIN_FINITE_PER_REPLICATE or len(i3_s) < MIN_FINITE_PER_REPLICATE:
        return None

    tr2_s, _ = train_test_split(i2_s, test_size=0.5, random_state=seed)
    tr3_s, _ = train_test_split(i3_s, test_size=0.5, random_state=seed)
    S_pool_s = np.vstack([seq_s[tr2_s], seq_s[tr3_s]])
    y_pool_s = np.concatenate([y2_s[tr2_s], y3_s[tr3_s]])
    w_pool_s = np.concatenate([w2_s[tr2_s], w3_s[tr3_s]])

    F_s, J_s, rank_s, _ = R.fit_weights_potts_from_data_matrixfree(
        S_pool_s, y_pool_s, sample_weight=w_pool_s, verbose=False, lam=0.0)
    return np.asarray(F_s), np.asarray(J_s), n_s, rank_s


def r_on_full_raw(F_, J_):
    s2 = score_FJ(seq_all[m2_all], F_, J_)
    s3 = score_FJ(seq_all[m3_all], F_, J_)
    return pearson(y2_all[m2_all], s2), pearson(y3_all[m3_all], s3)


final_rows = []

# (a) toutes les combinaisons de la grille plasmid_min x tolerance_frac (section 6)
for pm in PLASMID_MIN_GRID:
    for denom in TOLERANCE_DENOM_GRID:
        keep_s = build_population_mask(pm, denom)
        label = f"proportionnel plasmid_min={pm} tol=1/{denom}"
        out = fit_potts_weights_on_mask(keep_s)
        if out is None:
            print(f"{label:45s} n={int(keep_s.sum()):>7,}  -- trop peu de donnees, saute")
            final_rows.append(dict(filtre=label, n=int(keep_s.sum()), r_brut2=np.nan, r_brut3=np.nan))
            continue
        F_s, J_s, n_s, rank_s = out
        r_b2, r_b3 = r_on_full_raw(F_s, J_s)
        print(f"{label:45s} n={n_s:>7,}  r_brut org2/org3 = {r_b2:+.3f}/{r_b3:+.3f}")
        final_rows.append(dict(filtre=label, n=n_s, r_brut2=r_b2, r_brut3=r_b3))

# (b) fit principal de ce notebook (tol=1/5, section 2/3a) -- deja score sur le brut
final_rows.append(dict(filtre=f"PRINCIPAL proportionnel tol=1/5 ({tag_final})",
                        n=len(df_full), r_brut2=r_brut2, r_brut3=r_brut3))
print(f"{'PRINCIPAL proportionnel tol=1/5 (' + tag_final + ')':45s} n={len(df_full):>7,}  "
      f"r_brut org2/org3 = {r_brut2:+.3f}/{r_brut3:+.3f}")

# (c) fit readout-depth T=5 (AAV2_SEL_potts_readout_depth.ipynb), poids exportes dans lib/
F_T5_path = LIB / "aav2_F_sel_pool_potts_sorted_readoutT5_unreg.npy"
J_T5_path = LIB / "aav2_J_sel_pool_potts_sorted_readoutT5_unreg.npy"
if F_T5_path.exists() and J_T5_path.exists():
    F_T5, J_T5 = np.load(F_T5_path), np.load(J_T5_path)
    r_T5_2, r_T5_3 = r_on_full_raw(F_T5, J_T5)
    label_t5 = "readout-depth T=5 (AAV2_SEL_potts_readout_depth.ipynb)"
    final_rows.append(dict(filtre=label_t5, n=np.nan, r_brut2=r_T5_2, r_brut3=r_T5_3))
    print(f"{label_t5:45s} r_brut org2/org3 = {r_T5_2:+.3f}/{r_T5_3:+.3f}")
else:
    print("(poids readout-depth T=5 introuvables dans lib/ -- notebook pas encore relance)")

final_df = pd.DataFrame(final_rows)
final_df["r_brut_mean"] = final_df[["r_brut2", "r_brut3"]].mean(axis=1)
final_df = final_df.sort_values("r_brut_mean", ascending=False).reset_index(drop=True)
display(final_df)

best = final_df.iloc[0]
print(f"\n>>> Meilleure regression sur TOUT le dataset brut (org2>0 ET org3>0, n={int(m2_all.sum()):,}) : "
      f"{best['filtre']}  (r_brut_mean={best['r_brut_mean']:+.3f}, n_fit={best['n']:.0f})")